# Power Analysis: Should We Run This Experiment?

Power analysis is a **feasibility study**. Before committing 5,000 clients to a 90-day holdout, we need to know whether the experiment can actually detect the effect we care about — or whether we'd be burning calendar time for an inconclusive result.

### This notebook answers three questions:

| # | Question | Section | What a "no" means |
|---|----------|---------|--------------------|
| 1 | Do we have enough clients to detect a meaningful lift? | §2 Sample Sizing | Recruit more clients or find a better variance reduction strategy |
| 2 | What's the smallest lift we'd be able to detect? | §3 MDE | If MDE > expected effect, the experiment is blind to the signal |
| 3 | How long do we need to run? | §5 Duration | Extend runtime or accept lower power |

If any answer is "we can't", we redesign *before* burning 90 days of experiment time.

### Key concepts (plain English)

- **MDE (Minimum Detectable Effect):** The experiment's resolution limit. If the true lift is smaller than MDE, the experiment will almost certainly say "not significant" even though the effect is real. Think of it as trying to read 6pt font from across the room — the effect is there, but your instrument can't see it.

- **Power (1 − β):** The probability that *if the treatment actually works*, the experiment correctly detects it. 80% power means: if there's a real 10% lift, we'll catch it 4 out of 5 times. The other 1 in 5 times is a false negative.

- **α (significance level):** The probability of a false positive — declaring an effect when there is none. Set at 0.05 by convention.

- **CUPED:** A variance reduction technique that uses pre-experiment data to "subtract out" the predictable part of each client's behavior. The less noise, the smaller our MDE, the faster we reach a conclusion. Variance reduction = (1 − ρ²), where ρ is the correlation between pre- and post-experiment outcomes.

**The tradeoff:** For a fixed sample, you can't improve MDE, power, and α simultaneously. CUPED breaks this constraint by reducing noise — effectively giving you a larger sample for free.

---

**Structure:**
- **Part 1 (Prospective):** Before the experiment. Uses only historical data and planned sample sizes.
- **Part 2 (Post-Hoc Validation):** After the experiment. Checks whether our planning assumptions held up.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import sys
import IPython.display as display

import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path(".").resolve().parent / "src"))
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
from analysis.power_analysis import (
    minimum_detectable_effect,
    required_sample_size,
    power_at_effect_size,
    power_curve_with_cuped,
    simulate_power,
    experiment_sizing_table,
    duration_power_tradeoff,
    power_report,
)
from config import N_CLIENTS

data_dir = Path("..") / "data"
clients = pd.read_csv(data_dir / "raw" / "clients.csv")
pre_activity = pd.read_csv(data_dir / "raw" / "pre_period_activity.csv")

---
## Part 1 — Prospective: Planning the Experiment

Everything below uses **only pre-experiment data**. The experiment hasn't run yet — we're deciding whether it's worth running.

### §1. Planning Inputs: What Do We Know Before We Start?

Before any calculations, we need to establish the raw materials:
- **How many clients** are eligible? (This is our sample budget — we can't exceed it.)
- **How noisy is the metric?** (Higher variance → harder to detect small effects.)
- **How much can CUPED help?** (We estimate this from pre-period split-half correlation as a proxy for the pre-post ρ we'll actually get.)

In [ ]:
# --- BASELINE STATISTICS FROM PRE-PERIOD ---
# Aggregate each client's weekly volumes into their 13-week total.
# This mirrors the experiment metric (total volume over the experiment window).
pre_volumes = pre_activity.groupby("client_id")["weekly_volume"].sum()
baseline_mean = pre_volumes.mean()
baseline_std = pre_volumes.std()

# --- PLANNED SAMPLE SPLIT ---
# 50/50 randomization of our eligible pool.
n_t_planned = N_CLIENTS // 2   # 2,500 treatment
n_c_planned = N_CLIENTS - n_t_planned  # 2,500 control

# --- PROSPECTIVE ρ ESTIMATE ---
# Problem: CUPED needs ρ(pre, post), but the experiment hasn't run yet.
# Solution: Split the pre-period in half (weeks 1-6 vs 7-13) and compute
# the correlation between the two halves. If a client's volume is stable
# across two halves of the pre-period, it'll be stable across pre→post too.
# This is conservative — split-half uses ~6 weeks per half (more noise)
# vs the actual pre-post which will span two full 13-week windows.
pre_h1 = pre_activity[pre_activity["week"] <= 6].groupby("client_id")["weekly_volume"].sum()
pre_h2 = pre_activity[pre_activity["week"] > 6].groupby("client_id")["weekly_volume"].sum()
rho_prospective = pre_h1.corr(pre_h2)

# --- CUPED-ADJUSTED VARIANCE ---
# CUPED reduces variance by (1 - ρ²). The adjusted SD determines our MDE.
cuped_std_prospective = baseline_std * np.sqrt(1 - rho_prospective**2)
var_reduction_prospective = (1 - (cuped_std_prospective / baseline_std) ** 2) * 100

print("PRE-EXPERIMENT PLANNING INPUTS")
print("=" * 50)
print(f"Eligible clients:           {N_CLIENTS:,}")
print(f"Planned split:              {n_t_planned:,} treatment / {n_c_planned:,} control")
print(f"Baseline volume (13-wk):    ${baseline_mean:,.0f} ± ${baseline_std:,.0f}")
print(f"Prospective ρ (split-half): {rho_prospective:.3f}")
print(f"CUPED variance reduction:   {var_reduction_prospective:.0f}%")
print(f"CUPED-adjusted SD:          ${cuped_std_prospective:,.0f} (down from ${baseline_std:,.0f})")

### §2. What Effect Do We Expect?

Power analysis is only useful if we have a target effect size to plan around. Otherwise we're answering "can we detect *something*?" without knowing what *something* is.

**Our target: 10% lift in trading volume** (with 5% as the pessimistic floor).

Where does this come from?

| Evidence | Effect Size | Source |
|----------|------------|--------|
| Behavioral nudges in financial services | 5–15% lift in engagement | Industry benchmarks |
| Progress bars in gamification literature | 10–20% lift in task completion | Nunes & Dreze (2006) |
| Personalized savings projections (our nudge is targeted, not generic) | Higher end of range | Design rationale |

**Planning scenarios:**
- **Pessimistic (5% lift):** The nudge barely moves behavior. We still want to detect this.
- **Expected (10% lift):** Our primary planning target.
- **Optimistic (15%+ lift):** Easy to detect; not the constraint.

The key question for the next sections: **Is our MDE below 5%?** If yes, we can detect even the pessimistic scenario. If MDE is between 5–10%, we can only detect the expected case. If MDE > 10%, the experiment is likely a waste of time.

### §3. Do We Have Enough Clients?

For each target lift, how many clients would we need? The bars show required total N with and without CUPED. The orange line marks our eligible pool (N=5,000).

**Reading the chart:** If a bar exceeds the orange line, we can't detect that effect with our available clients — we'd need to recruit more or find a better variance reduction strategy.

In [ ]:
target_lifts = [0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25]
sizing_rows = []

for lift in target_lifts:
    raw = required_sample_size(baseline_mean * lift, baseline_std)
    cuped = required_sample_size(baseline_mean * lift, cuped_std_prospective)
    sizing_rows.append({
        'lift_pct': lift * 100,
        'n_raw': raw['n_total'],
        'n_cuped': cuped['n_total'],
        'savings': raw['n_total'] - cuped['n_total']
    })

sizing_df = pd.DataFrame(sizing_rows)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[f"{r['lift_pct']:.0f}%" for _, r in sizing_df.iterrows()],
    y=sizing_df['n_raw'],
    name='Without CUPED',
    marker_color='#EF553B',
    text=sizing_df['n_raw'],
    textposition='outside',
    hovertemplate='%{x} lift<br>Need n=%{y:,}<extra>Raw</extra>'
))

fig.add_trace(go.Bar(
    x=[f"{r['lift_pct']:.0f}%" for _, r in sizing_df.iterrows()],
    y=sizing_df['n_cuped'],
    name='With CUPED (prospective ρ)',
    marker_color='#636EFA',
    text=sizing_df['n_cuped'],
    textposition='outside',
    hovertemplate='%{x} lift<br>Need n=%{y:,}<extra>CUPED</extra>'
))

fig.add_hline(y=N_CLIENTS, line_dash='dash', line_color='orange', line_width=2,
              annotation_text=f'Our eligible pool: N={N_CLIENTS:,}',
              annotation_position='top right')

fig.update_layout(
    title='Required Sample Size to Detect Each Effect (80% power, α=0.05)',
    xaxis_title='Minimum Detectable Lift',
    yaxis_title='Total Clients Needed',
    yaxis=dict(type='log'),
    template='plotly_white',
    barmode='group',
    width=850, height=480,
    legend=dict(x=0.7, y=0.95),
    margin=dict(t=60)
)

fig.show()

# Decision summary
print('\nFEASIBILITY CHECK')
print('=' * 70)
for _, row in sizing_df.iterrows():
    feasible_raw = '✓' if row['n_raw'] <= N_CLIENTS else '✗'
    feasible_cuped = '✓' if row['n_cuped'] <= N_CLIENTS else '✗'
    print(f"{row['lift_pct']:5.0f}% lift: need {row['n_raw']:>8,} raw ({feasible_raw}) | {row['n_cuped']:>6,} CUPED ({feasible_cuped})")

# Business punchline
cuped_5pct = sizing_df[sizing_df['lift_pct'] == 5.0]['n_cuped'].values[0]
raw_5pct = sizing_df[sizing_df['lift_pct'] == 5.0]['n_raw'].values[0]
print(f"\n→ Without CUPED, detecting a 5% lift requires {raw_5pct:,} clients — "
      f"{'exceeds' if raw_5pct > N_CLIENTS else 'within'} our pool.")
print(f"→ With CUPED, it requires only {cuped_5pct:,}. "
      f"CUPED isn't optional here — it's what makes the experiment feasible.")

In [ ]:
fig.write_image("../reports/figures/sample_size_plot.png", scale=2)
display.display(display.Image("../reports/figures/sample_size_plot.png"))

### §4. MDE: What's the Smallest Effect We Can See?

Given our planned N=5,000 (2,500 per arm) and CUPED with prospective ρ, what's the smallest lift that would be statistically significant 80% of the time?

**This is the planning MDE** — the effect size threshold below which our experiment is essentially blind. We need this to be below our pessimistic scenario (5% lift) to feel confident about proceeding.

In [ ]:
mde_raw = minimum_detectable_effect(n_t_planned, n_c_planned, baseline_std)
mde_cuped = minimum_detectable_effect(n_t_planned, n_c_planned, cuped_std_prospective)

print("MINIMUM DETECTABLE EFFECT (PROSPECTIVE)")
print("=" * 72)
print(f'{"Method":<25}{"MDE ($)":>15}    {"MDE (% lift)":>15} {"SE ($)":>15}')
print("-" * 72)
print(f'{"Raw":<25} ${mde_raw["mde_absolute"]:,.0f} '
      f'{mde_raw["mde_absolute"]/baseline_mean*100:>14.1f}% '
      f'${mde_raw["se"]:>14,.0f}')
print(f'{"CUPED (ρ=" + f"{rho_prospective:.3f})":<25} ${mde_cuped["mde_absolute"]:,.0f} '
      f'{mde_cuped["mde_absolute"]/baseline_mean*100:>14.1f}% '
      f'${mde_cuped["se"]:>14,.0f}')

mde_cuped_pct = mde_cuped['mde_absolute'] / baseline_mean * 100
print(f'\nCUPED reduces MDE by '
      f'{(1 - mde_cuped["mde_absolute"]/mde_raw["mde_absolute"])*100:.0f}% '
      f'(variance reduction: {var_reduction_prospective:.0f}%)')

# Connect back to expected effect
print(f'\n→ With CUPED, we can detect lifts as small as {mde_cuped_pct:.1f}%.')
if mde_cuped_pct < 5:
    print(f'  This is below our pessimistic scenario (5%). We are well-powered.')
elif mde_cuped_pct < 10:
    print(f'  This is below our expected scenario (10%) but above pessimistic (5%).')
    print(f'  We can detect the expected effect but may miss a subtle one.')
else:
    print(f'  ⚠ This exceeds our expected scenario (10%). Consider redesigning.')

### §5. Sensitivity to ρ: What If Market Conditions Shift?

Our entire design depends on ρ ≈ {rho_prospective:.3f}. But ρ is estimated from a stable pre-period. What happens if:
- A market regime shift (bull → bear) weakens the pre-post correlation?
- New client onboarding changes the composition of our sample?

This section stress-tests the design: **how bad does ρ need to get before our experiment becomes underpowered?**

In [ ]:
# Stress-test: MDE and power across a range of ρ values
rho_scenarios = [0.0, 0.3, 0.5, 0.7, 0.8, 0.9, rho_prospective, 0.99]
rho_scenarios = sorted(set([round(r, 3) for r in rho_scenarios]))

sensitivity_rows = []
for rho in rho_scenarios:
    adj_std = baseline_std * np.sqrt(1 - rho**2)
    mde_info = minimum_detectable_effect(n_t_planned, n_c_planned, adj_std)
    mde_pct = mde_info['mde_absolute'] / baseline_mean * 100
    var_red = (1 - (adj_std / baseline_std)**2) * 100
    
    # Can we detect 5% lift? 10% lift?
    power_5 = power_at_effect_size(baseline_mean * 0.05, n_t_planned, n_c_planned, adj_std)
    power_10 = power_at_effect_size(baseline_mean * 0.10, n_t_planned, n_c_planned, adj_std)
    
    is_ours = '  ← our estimate' if abs(rho - rho_prospective) < 0.001 else ''
    
    sensitivity_rows.append({
        'rho': rho, 'var_reduction': var_red, 'mde_pct': mde_pct,
        'power_at_5pct': power_5, 'power_at_10pct': power_10, 'label': is_ours
    })

sens_df = pd.DataFrame(sensitivity_rows)

print("SENSITIVITY ANALYSIS: How robust is our design to ρ dropping?")
print("=" * 85)
print(f'{"ρ":<8} {"Var Reduction":>14} {"MDE (% lift)":>13} '
      f'{"Power @5%":>11} {"Power @10%":>12}  {""}')
print("-" * 85)
for _, row in sens_df.iterrows():
    flag = ''
    if row['power_at_10pct'] < 0.80:
        flag = '⚠ underpowered for 10% lift'
    print(f'{row["rho"]:<8.3f} {row["var_reduction"]:>13.0f}% {row["mde_pct"]:>12.1f}% '
          f'{row["power_at_5pct"]:>11.0%} {row["power_at_10pct"]:>11.0%}  '
          f'{row["label"]}{flag}')

# Visualize
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['MDE vs ρ', 'Power at 10% Lift vs ρ'],
    horizontal_spacing=0.12
)

fig.add_trace(go.Scatter(
    x=sens_df['rho'], y=sens_df['mde_pct'],
    mode='lines+markers', name='MDE',
    line=dict(color='#636EFA', width=2.5),
    marker=dict(size=7),
), row=1, col=1)

# Mark our estimate
our_row = sens_df[sens_df['label'] != ''].iloc[0]
fig.add_trace(go.Scatter(
    x=[our_row['rho']], y=[our_row['mde_pct']],
    mode='markers', name='Our estimate',
    marker=dict(color='orange', size=14, symbol='star'),
    showlegend=True
), row=1, col=1)

# Target effect lines
fig.add_hline(y=5, line_dash='dot', line_color='green', row=1, col=1,
              annotation_text='Pessimistic (5%)', annotation_position='bottom right')
fig.add_hline(y=10, line_dash='dot', line_color='gray', row=1, col=1,
              annotation_text='Expected (10%)', annotation_position='top right')

fig.add_trace(go.Scatter(
    x=sens_df['rho'], y=sens_df['power_at_10pct'],
    mode='lines+markers', name='Power @10%',
    line=dict(color='#EF553B', width=2.5),
    marker=dict(size=7),
), row=1, col=2)

fig.add_hline(y=0.80, line_dash='dot', line_color='gray', row=1, col=2,
              annotation_text='80% power', annotation_position='bottom right')

fig.update_xaxes(title_text='Pre-Post Correlation (ρ)', row=1, col=1)
fig.update_xaxes(title_text='Pre-Post Correlation (ρ)', row=1, col=2)
fig.update_yaxes(title_text='MDE (% lift)', row=1, col=1)
fig.update_yaxes(title_text='Power', tickformat='.0%', row=1, col=2)

fig.update_layout(
    title='Sensitivity: How Robust Is Our Design If ρ Drops?',
    template='plotly_white', width=950, height=450,
    showlegend=True,
    legend=dict(x=0.02, y=0.98),
    margin=dict(t=60)
)
fig.show()

print(f"\n→ Even if ρ drops to 0.70 (a severe regime shift), "
      f"MDE stays at {sens_df[sens_df['rho']==0.7]['mde_pct'].values[0]:.1f}%.")
print(f"  The design is robust to moderate ρ degradation.")
print(f"  At ρ = 0.0 (CUPED completely fails), we fall back to the raw estimator "
      f"with MDE = {sens_df[sens_df['rho']==0.0]['mde_pct'].values[0]:.1f}%.")

In [ ]:
fig.write_image("../reports/figures/rho_sensitivity.png", scale=2)
display.display(display.Image("../reports/figures/rho_sensitivity.png"))

### §6. Power Curves: The Full Picture

The power curve answers: **for each possible true effect size, what's the probability our experiment detects it?**

The vertical dashed lines mark each method's MDE (the effect size where power crosses 80%). CUPED shifts the entire curve left — we detect smaller effects at the same power level.

In [ ]:
curves = power_curve_with_cuped(
    baseline_std, n_t_planned, n_c_planned, rho_prospective, baseline_mean
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=curves['pct_lift'], y=curves['power_raw'],
    mode='lines', name='Raw',
    line=dict(color='#EF553B', width=2.5),
    hovertemplate='Effect: %{x:.1f}% lift<br>Power: %{y:.1%}<extra>Raw</extra>',
))

fig.add_trace(go.Scatter(
    x=curves['pct_lift'], y=curves['power_cuped'],
    mode='lines', name=f'CUPED (prospective ρ={rho_prospective:.3f})',
    line=dict(color='#636EFA', width=2.5),
    hovertemplate='Effect: %{x:.1f}% lift<br>Power: %{y:.1%}<extra>CUPED</extra>',
))

fig.add_hline(y=0.80, line_dash='dot', line_color='gray',
              annotation_text='80% power', annotation_position='bottom right')

mde_raw_pct = curves.attrs['mde_raw'] / baseline_mean * 100
mde_cuped_pct = curves.attrs['mde_cuped'] / baseline_mean * 100

fig.add_vline(x=mde_raw_pct, line_dash='dash', line_color='#EF553B', line_width=1,
              annotation_text=f'Raw MDE: {mde_raw_pct:.1f}%',
              annotation_position='top right', annotation_font_color='#EF553B')
fig.add_vline(x=mde_cuped_pct, line_dash='dash', line_color='#636EFA', line_width=1,
              annotation_text=f'CUPED MDE: {mde_cuped_pct:.1f}%',
              annotation_position='top left', annotation_font_color='#636EFA')

fig.update_layout(
    title='Power Curve: Raw vs CUPED (Prospective Planning)',
    xaxis_title='True Effect Size (% lift over baseline volume)',
    yaxis_title='Statistical Power (probability of detection)',
    yaxis=dict(range=[0, 1.05], tickformat='.0%'),
    template='plotly_white',
    width=850, height=500,
    legend=dict(x=0.7, y=0.3),
    margin=dict(t=60),
)

fig.show()

In [ ]:
fig.write_image("../reports/figures/power_curve.png", scale=2)
display.display(display.Image("../reports/figures/power_curve.png"))

### §7. Duration: Is 13 Weeks Enough?

Longer experiments reduce within-client noise (weekly fluctuations average out), but they also cost more in holdout opportunity cost — control clients miss the intervention for longer.

**The variance model:** Var(W-week volume) = W² · σ²_between + W · σ²_within.
- Between-client variance (some trade $500M, others $10M) scales with W² — it's who the client *is*.
- Within-client noise (same client trades $50M one week, $80M the next) scales with W — it averages out.

So duration mainly helps when within-client noise is large relative to between-client differences.

In [ ]:
duration = duration_power_tradeoff(
    pre_activity, n_t_planned, n_c_planned, effect_pct=0.10
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Power vs Duration (at 10% lift)', 'MDE vs Duration'],
    horizontal_spacing=0.12,
)

fig.add_trace(go.Scatter(
    x=duration['weeks'], y=duration['power_at_target_effect'],
    mode='lines+markers', name='Power',
    line=dict(color='#636EFA', width=2.5), marker=dict(size=5),
    hovertemplate='Week %{x}<br>Power: %{y:.1%}<extra></extra>',
), row=1, col=1)

fig.add_hline(y=0.80, line_dash='dot', line_color='gray', row=1, col=1)
fig.add_vline(x=13, line_dash='dash', line_color='orange', line_width=1.5,
              annotation_text='Planned: 13 weeks', row=1, col=1)

fig.add_trace(go.Scatter(
    x=duration['weeks'], y=duration['mde_pct'],
    mode='lines+markers', name='MDE (%)',
    line=dict(color='#EF553B', width=2.5), marker=dict(size=5),
    hovertemplate='Week %{x}<br>MDE: %{y:.1f}%<extra></extra>',
), row=1, col=2)

fig.add_vline(x=13, line_dash='dash', line_color='orange', line_width=1.5, row=1, col=2)

fig.update_yaxes(title_text='Power', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='MDE (% lift)', row=1, col=2)
fig.update_xaxes(title_text='Experiment Duration (weeks)', row=1, col=1)
fig.update_xaxes(title_text='Experiment Duration (weeks)', row=1, col=2)

fig.update_layout(
    template='plotly_white', width=950, height=420,
    showlegend=False, margin=dict(t=60)
)
fig.show()

week13 = duration[duration['weeks'] == 13].iloc[0]
print(f'At 13 weeks: Power = {week13["power_at_target_effect"]:.0%}, '
      f'MDE = {week13["mde_pct"]}%')
print(f'→ 13 weeks is sufficient for a 10% lift. Proceeding with one-quarter duration.')

### §8. Subgroup Power: Where We'll Be Blind

Before we run, we should know which subgroup analyses to trust and which will be underpowered noise.

**The mistake junior DSs make:** Slice data into subgroups, run tests, and report "not significant" as "no effect." But if a subgroup has n=50, it can't detect anything below a 30% lift. "Not significant" just means "too little data" — not "no effect."

This chart shows **prospectively** which segment results will be trustworthy. Bars that don't reach the 80% threshold should be treated with caution when interpreting results.

In [ ]:
# Merge pre-period volume onto client demographics
client_vols = clients[['client_id', 'segment', 'distance_bucket']].copy()
client_vols['pre_volume'] = client_vols['client_id'].map(pre_volumes)

# Compute subgroup power prospectively (planned 50/50 split — no group assignments yet)
sub_rows = []
for col in ['distance_bucket', 'segment']:
    for subgroup, sub_df in client_vols.groupby(col):
        n_sub = len(sub_df)
        n_t_sub = n_sub // 2
        n_c_sub = n_sub - n_t_sub
        sub_std = sub_df['pre_volume'].std()
        sub_mean = sub_df['pre_volume'].mean()

        if n_t_sub < 5 or n_c_sub < 5:
            continue

        mde_info = minimum_detectable_effect(n_t_sub, n_c_sub, sub_std)
        mde_pct = mde_info['mde_absolute'] / sub_mean * 100 if sub_mean > 0 else np.nan

        power_10 = power_at_effect_size(sub_mean * 0.10, n_t_sub, n_c_sub, sub_std)
        power_15 = power_at_effect_size(sub_mean * 0.15, n_t_sub, n_c_sub, sub_std)
        power_25 = power_at_effect_size(sub_mean * 0.25, n_t_sub, n_c_sub, sub_std)

        sub_rows.append({
            'cut': col, 'subgroup': subgroup, 'n_total': n_sub,
            'mde_pct': round(mde_pct, 2),
            'power_at_10pct_lift': round(power_10, 3),
            'power_at_15pct_lift': round(power_15, 3),
            'power_at_25pct_lift': round(power_25, 3),
            'adequately_powered': 'Yes' if mde_pct < 20 else 'No'
        })

all_subs = pd.DataFrame(sub_rows)
dist_df = all_subs[all_subs['cut'] == 'distance_bucket']
seg_df  = all_subs[all_subs['cut'] == 'segment']

LIFT_SERIES = [
    ('power_at_10pct_lift', '10% lift', '#EF553B'),
    ('power_at_15pct_lift', '15% lift', '#FFA15A'),
    ('power_at_25pct_lift', '25% lift', '#00CC96'),
]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['By Distance to Next Tier', 'By Client Segment'],
    horizontal_spacing=0.14,
)

for df_sub, col_idx in [(dist_df, 1), (seg_df, 2)]:
    for lift_col, lift_label, color in LIFT_SERIES:
        fig.add_trace(go.Bar(
            y=df_sub['subgroup'], x=df_sub[lift_col],
            name=lift_label, orientation='h', marker_color=color,
            showlegend=(col_idx == 1),
            hovertemplate='%{y}<br>Power: %{x:.1%}<extra>' + lift_label + '</extra>',
        ), row=1, col=col_idx)

for col_idx in [1, 2]:
    fig.add_vline(x=0.80, line_dash='dash', line_color='grey', line_width=1.5,
                  row=1, col=col_idx)

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(dash='dash', color='grey', width=1.5),
    name='80% threshold', showlegend=True,
))

fig.update_xaxes(range=[0, 1.0], tickformat='.0%', title_text='Statistical Power')
fig.update_layout(
    title='Prospective Subgroup Power (Planned 50/50 Split)',
    template='plotly_white', barmode='group',
    width=1050, height=500,
    legend=dict(orientation='h', x=0.28, y=-0.18),
    margin=dict(t=80, b=100, l=140),
)
fig.show()

# Print tables
for cut_name in ['distance_bucket', 'segment']:
    cut_df = all_subs[all_subs['cut'] == cut_name]
    print(f'\n=== {cut_name.replace("_", " ").title()} Power ===')
    print(cut_df[['subgroup', 'n_total', 'mde_pct', 'power_at_10pct_lift',
                   'power_at_15pct_lift', 'power_at_25pct_lift',
                   'adequately_powered']].to_string(index=False))

# Warnings
underpowered = all_subs[all_subs['adequately_powered'] == 'No']
if len(underpowered) > 0:
    print('\n⚠ UNDERPOWERED SUBGROUPS — treat results with caution:')
    for _, row in underpowered.iterrows():
        print(f"  {row['subgroup']}: MDE = {row['mde_pct']:.0f}% (n={row['n_total']})")
else:
    print('\n→ All subgroups are adequately powered (MDE < 20%). Proceeding.')

### §9. Sizing Grid for Leadership

Leadership asks: *"We want 80% power to detect a 10% lift — how many clients do we need?"* This heatmap answers that for every combination of sample size and effect size.

**How to read it:** Find your desired lift on the x-axis, then read down to find the smallest N that gives ≥80% power (green cells).

In [ ]:
sizing = experiment_sizing_table(baseline_std, baseline_mean)

power_cols = [c for c in sizing.columns if c.startswith('power_at_')]
lift_labels = [c.replace('power_at_', '').replace('pct_lift', '% lift') for c in power_cols]
z_data = sizing[power_cols].values

fig = go.Figure(data=go.Heatmap(
    z=z_data,
    x=lift_labels,
    y=[f"n={n} (MDE={m}%)" for n, m in zip(sizing['n_total'], sizing['mde_pct'])],
    colorscale='Darkmint',
    zmin=0, zmax=1,
    text=[[f"{v:.0%}" for v in row] for row in z_data],
    texttemplate='%{text}',
    textfont=dict(size=13),
    hovertemplate='Sample size: %{y}<br>Effect: %{x}<br>Power: %{z:.1%}<extra></extra>',
    colorbar=dict(title='Power', tickformat='.0%'),
))

fig.update_layout(
    title='Experiment Sizing Grid: Power at Each (Sample Size × Effect Size)',
    xaxis_title='True Effect Size',
    yaxis_title='Total Sample Size (with MDE at 80% power)',
    template='plotly_white',
    width=800, height=450,
    margin=dict(t=60, l=180),
)
fig.show()

print(f'\nOur planned experiment uses N={N_CLIENTS:,}.')

In [ ]:
fig.write_image("../reports/figures/heatmap.png", scale=2)
display.display(display.Image("../reports/figures/heatmap.png"))

### Prospective Summary: Go / No-Go Decision

Before moving to the post-hoc section, let's consolidate the planning conclusions.

In [ ]:
print("PROSPECTIVE GO / NO-GO")
print("=" * 60)
print(f"Expected effect:        10% lift (pessimistic: 5%)")
print(f"MDE with CUPED:         {mde_cuped_pct:.1f}% lift")
print(f"MDE without CUPED:      {mde_raw_pct:.1f}% lift")
print(f"Planned duration:       13 weeks (power = {week13['power_at_target_effect']:.0%} at 10% lift)")
print(f"ρ sensitivity:          Design is robust down to ρ ≈ 0.5")
print(f"Underpowered subgroups: {len(underpowered)}")
print()
if mde_cuped_pct < 5:
    print("✅ DECISION: GO. MDE is below pessimistic scenario. Experiment is well-powered.")
elif mde_cuped_pct < 10:
    print("✅ DECISION: GO (with caveat). MDE is below expected scenario but above pessimistic.")
    print("   We can detect moderate effects but may miss very subtle ones.")
else:
    print("❌ DECISION: REDESIGN. MDE exceeds expected effect. Experiment is likely to be inconclusive.")

---

## Part 2 — Post-Hoc: Was Our Design Adequate?

The experiment has run for 13 weeks. This section **validates our planning assumptions**, not the experiment results.

We're checking:
- Was split-half ρ a good proxy for actual pre-post ρ?
- Was our analytical power formula reasonably calibrated against simulation?

**Note on post-hoc power:** Post-hoc power analysis is controversial in statistics. Some argue it's circular: significant results always have high post-hoc power, and vice versa. Our use here is different — we're auditing the *planning process*, not reinterpreting results.

### §10. Planned vs Actual Parameters

In [ ]:
# Load actual experiment results
df = pd.read_csv(data_dir / 'processed' / 'experiment_summary.csv')

n_t_actual = (df['group'] == 'treatment').sum()
n_c_actual = (df['group'] == 'control').sum()

# Post-hoc ρ: the actual pre-post correlation (only available after experiment)
rho_posthoc = df[['pre_volume', 'post_volume']].corr().iloc[0, 1]
cuped_std_posthoc = baseline_std * np.sqrt(1 - rho_posthoc**2)
var_reduction_posthoc = (1 - (cuped_std_posthoc / baseline_std)**2) * 100

print('PLANNED vs ACTUAL')
print('=' * 56)
print(f'{"Parameter":<30} {"Planned":>12} {"Actual":>12}')
print('-' * 56)
print(f'{"Treatment N":<30} {n_t_planned:>12,} {n_t_actual:>12,}')
print(f'{"Control N":<30} {n_c_planned:>12,} {n_c_actual:>12,}')
print(f'{"ρ (pre-post)":<30} {rho_prospective:>12.3f} {rho_posthoc:>12.3f}')
print(f'{"CUPED variance reduction":<30} {var_reduction_prospective:>11.0f}% {var_reduction_posthoc:>11.0f}%')
print()
print(f'→ Prospective split-half ρ ({rho_prospective:.3f}) slightly '
      f'{"under" if rho_prospective < rho_posthoc else "over"}estimates '
      f'actual ρ ({rho_posthoc:.3f}).')
print(f'  This is expected: split-half uses ~6 weeks per half (noisier) vs '
      f'the full 13-week pre-post span.')

### §11. MDE Comparison and Power Curve Validation

Side-by-side comparison of prospective vs post-hoc MDE and power curves. If the dashed (post-hoc) curve is close to the solid (prospective) curve, our planning was sound.

In [ ]:
# MDE comparison across all three methods
mde_raw_actual = minimum_detectable_effect(n_t_actual, n_c_actual, baseline_std)
mde_cuped_prospective = minimum_detectable_effect(n_t_planned, n_c_planned, cuped_std_prospective)
mde_cuped_posthoc = minimum_detectable_effect(n_t_actual, n_c_actual, cuped_std_posthoc)

print('MDE COMPARISON: PROSPECTIVE vs POST-HOC')
print('=' * 64)
print(f'{"Method":<35} {"MDE ($)":>15} {"MDE (% lift)":>12}')
print('-' * 64)
print(f'{"Raw (planned N=5000)":.<35} ${mde_raw["mde_absolute"]:>14,.0f} '
      f'{mde_raw["mde_absolute"]/baseline_mean*100:>11.1f}%')
print(f'{"CUPED prospective (ρ=" + f"{rho_prospective:.3f})":.<35} '
      f'${mde_cuped_prospective["mde_absolute"]:>14,.0f} '
      f'{mde_cuped_prospective["mde_absolute"]/baseline_mean*100:>11.1f}%')
print(f'{"CUPED post-hoc (ρ=" + f"{rho_posthoc:.3f})":.<35} '
      f'${mde_cuped_posthoc["mde_absolute"]:>14,.0f} '
      f'{mde_cuped_posthoc["mde_absolute"]/baseline_mean*100:>11.1f}%')

# Three-line power curve comparison
curves_prosp = power_curve_with_cuped(
    baseline_std, n_t_planned, n_c_planned, rho_prospective, baseline_mean)
curves_posthoc = power_curve_with_cuped(
    baseline_std, n_t_actual, n_c_actual, rho_posthoc, baseline_mean)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=curves_prosp['pct_lift'], y=curves_prosp['power_raw'],
    mode='lines', name='Raw',
    line=dict(color='#EF553B', width=2.5),
))
fig.add_trace(go.Scatter(
    x=curves_prosp['pct_lift'], y=curves_prosp['power_cuped'],
    mode='lines', name=f'CUPED prospective (ρ={rho_prospective:.3f})',
    line=dict(color='#636EFA', width=2.5),
))
fig.add_trace(go.Scatter(
    x=curves_posthoc['pct_lift'], y=curves_posthoc['power_cuped'],
    mode='lines', name=f'CUPED post-hoc (ρ={rho_posthoc:.3f})',
    line=dict(color='#636EFA', width=2.5, dash='dash'),
))

fig.add_hline(y=0.80, line_dash='dot', line_color='gray',
              annotation_text='80% power', annotation_position='bottom right')

mde_prosp_pct = curves_prosp.attrs['mde_cuped'] / baseline_mean * 100
mde_posthoc_pct = curves_posthoc.attrs['mde_cuped'] / baseline_mean * 100

fig.add_vline(x=mde_prosp_pct, line_dash='dash', line_color='#636EFA', line_width=1,
              annotation_text=f'Prospective MDE: {mde_prosp_pct:.1f}%',
              annotation_position='top right', annotation_font_color='#636EFA')
fig.add_vline(x=mde_posthoc_pct, line_dash='dot', line_color='#636EFA', line_width=1,
              annotation_text=f'Post-hoc MDE: {mde_posthoc_pct:.1f}%',
              annotation_position='bottom left', annotation_font_color='#636EFA')

fig.update_layout(
    title='Power Curves: Prospective Planning vs Post-Hoc Validation',
    xaxis_title='True Effect Size (% lift)',
    yaxis_title='Statistical Power',
    yaxis=dict(range=[0, 1.05], tickformat='.0%'),
    template='plotly_white',
    width=850, height=500,
    legend=dict(x=0.55, y=0.25),
    margin=dict(t=60)
)
fig.show()

print(f'\n→ Prospective MDE ({mde_prosp_pct:.1f}%) was slightly conservative '
      f'vs post-hoc ({mde_posthoc_pct:.1f}%), as expected.')

### §12. Simulation vs Analytical: The Reality Check

Analytical formulas assume the sampling distribution of the mean is Normal (via CLT). Trading volume is log-normal with heavy tails — a few Market Makers trading $500M+ inflate variance massively. The analytical formulas may therefore **overstate power** because:
- The t-test's p-values are unreliable when distributions are this skewed.
- A single whale landing in treatment vs control can flip the result.
- The "80% power" the formula promises might actually be 60% in practice.

Simulation-based power uses the **actual** volume distribution and gives a more honest estimate.

**Reading the chart:** Dashed lines = analytical formula. Dots = simulation with 95% CI. If dots fall below dashed lines, the analytical formulas are overestimating power and we need to adjust our design accordingly.

In [ ]:
baseline_volumes = df['pre_volume'].values
pre_vols_for_cuped = df['pre_volume'].values

effect_sizes = [0.05, 0.10, 0.15, 0.20, 0.25]
sim_results = []

for eff in effect_sizes:
    sim_raw = simulate_power(
        baseline_volumes, eff, n_t_actual, n_c_actual,
        n_simulations=1000, seed=42)
    sim_cuped = simulate_power(
        baseline_volumes, eff, n_t_actual, n_c_actual,
        n_simulations=1000, seed=42,
        use_cuped=True, pre_volumes=pre_vols_for_cuped)
    analytical_raw = power_at_effect_size(
        baseline_mean * eff, n_t_actual, n_c_actual, baseline_std)
    analytical_cuped = power_at_effect_size(
        baseline_mean * eff, n_t_actual, n_c_actual, cuped_std_posthoc)

    sim_results.append({
        'effect_pct': eff * 100,
        'analytical_raw': analytical_raw,
        'sim_raw': sim_raw['power'],
        'sim_raw_se': sim_raw['se_of_power'],
        'analytical_cuped': analytical_cuped,
        'sim_cuped': sim_cuped['power'],
        'sim_cuped_se': sim_cuped['se_of_power'],
    })

sim_df = pd.DataFrame(sim_results)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=sim_df['effect_pct'], y=sim_df['analytical_raw'],
    mode='lines', name='Analytical (raw)',
    line=dict(color='#EF553B', width=2, dash='dash'),
))
fig.add_trace(go.Scatter(
    x=sim_df['effect_pct'], y=sim_df['analytical_cuped'],
    mode='lines', name='Analytical (CUPED)',
    line=dict(color='#636EFA', width=2, dash='dash'),
))
fig.add_trace(go.Scatter(
    x=sim_df['effect_pct'], y=sim_df['sim_raw'],
    mode='markers', name='Simulated (raw)',
    marker=dict(color='#EF553B', size=10, symbol='circle'),
    error_y=dict(type='data', array=sim_df['sim_raw_se'] * 1.96, visible=True),
    hovertemplate='%{x}% lift<br>Sim power: %{y:.1%}<extra>Sim (raw)</extra>'
))
fig.add_trace(go.Scatter(
    x=sim_df['effect_pct'], y=sim_df['sim_cuped'],
    mode='markers', name='Simulated (CUPED)',
    marker=dict(color='#636EFA', size=10, symbol='diamond'),
    error_y=dict(type='data', array=sim_df['sim_cuped_se'] * 1.96, visible=True),
    hovertemplate='%{x}% lift<br>Sim power: %{y:.1%}<extra>Sim (CUPED)</extra>'
))

fig.add_hline(y=0.80, line_dash='dot', line_color='gray')

fig.update_layout(
    title='Analytical vs Simulation-Based Power (1,000 Monte Carlo iterations)',
    xaxis_title='True Effect Size (% lift)',
    yaxis_title='Power',
    yaxis=dict(range=[0, 1.05], tickformat='.0%'),
    template='plotly_white',
    width=850, height=500,
    legend=dict(x=0.55, y=0.25),
    margin=dict(t=60)
)
fig.show()

### §13. Full Power Report Summary

In [ ]:
report = power_report(df, pre_activity)

print('=' * 60)
print('POWER ANALYSIS SUMMARY')
print('=' * 60)
s = report['summary']
print(f"Sample:              {s['n_treatment']} treatment, {s['n_control']} control")
print(f"Baseline volume:     ${s['baseline_mean']:,.0f} ± ${s['baseline_std']:,.0f}")
print(f"MDE (raw):           {s['mde_raw_pct']:.1f}% lift")
print(f"MDE (CUPED):         {s['mde_cuped_pct']:.1f}% lift")
print(f"Pre-post ρ:          {report['pre_post_correlation']:.3f}")
print(f"CUPED var reduction: {report['variance_reduction_from_cuped_pct']:.0f}%")
print()

if report['underpowered_subgroups']:
    print('⚠ UNDERPOWERED SUBGROUPS (treat results with caution):')
    for msg in report['underpowered_subgroups']:
        print(msg)
else:
    print('All subgroups are adequately powered (MDE < 20%).')

print()
print('BOTTOM LINE: ', end='')
if s['mde_cuped_pct'] < 10:
    print(f'With CUPED, we can detect lifts as small as {s["mde_cuped_pct"]:.1f}%. '
          f'Design is adequate.')
elif s['mde_raw_pct'] < 15:
    print(f'Without CUPED, we can detect ~{s["mde_raw_pct"]:.0f}% lifts. '
          f'CUPED strongly recommended.')
else:
    print(f'MDE is {s["mde_raw_pct"]:.0f}% — we can only detect large effects. '
          f'Consider more clients.')

print(f'\nNote: Post-hoc figures use actual ρ = {rho_posthoc:.3f}. '
      f'Prospective ρ (split-half) = {rho_prospective:.3f}.')